In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


# read opration 
df = spark.read.format('csv')\
                .option("header", True)\
                .option("inferSchema", True)\
                .load('abfss://SusilNayakWS@onelake.dfs.fabric.microsoft.com/bronze_lh.Lakehouse/Files/RawData/olist_customers_dataset.csv')
display(df.limit(7))

StatementMeta(, 6733142a-fee5-435e-80a4-e80ba8830b9c, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d2bb2536-e2cc-4df5-88ad-985e9f6f4257)

In [2]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

StatementMeta(, 6733142a-fee5-435e-80a4-e80ba8830b9c, 4, Finished, Available, Finished, False)

### Select Columns

In [3]:
df = df.select("customer_id", "customer_unique_id","customer_city", "customer_zip_code_prefix")

StatementMeta(, 6733142a-fee5-435e-80a4-e80ba8830b9c, 5, Finished, Available, Finished, False)

### Rename column

In [4]:
df = df.withColumnRenamed("customer_unique_id", "unique_id")\
        .withColumnRenamed("customer_zip_code_prefix", "zipcode")

StatementMeta(, 6733142a-fee5-435e-80a4-e80ba8830b9c, 6, Finished, Available, Finished, False)

In [5]:
display(df.limit(5))

StatementMeta(, 6733142a-fee5-435e-80a4-e80ba8830b9c, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, aecf1949-6aa9-46cf-807d-5473847c2571)

## Typecasting

In [6]:
df = df.withColumn("zipcode", col("zipcode").cast(IntegerType()))

StatementMeta(, 6733142a-fee5-435e-80a4-e80ba8830b9c, 8, Finished, Available, Finished, False)

In [7]:
df.printSchema()

StatementMeta(, 6733142a-fee5-435e-80a4-e80ba8830b9c, 9, Finished, Available, Finished, False)

root
 |-- customer_id: string (nullable = true)
 |-- unique_id: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- zipcode: integer (nullable = true)



In [8]:
df_customers = df
# load order items

df_orderitems = spark.read.format('csv')\
                            .option("header", True)\
                            .option("inferSchema", True)\
                            .load("abfss://SusilNayakWS@onelake.dfs.fabric.microsoft.com/bronze_lh.Lakehouse/Files/RawData/olist_order_items_dataset.csv")

StatementMeta(, 6733142a-fee5-435e-80a4-e80ba8830b9c, 10, Finished, Available, Finished, False)

In [9]:
display(df_orderitems.limit(6))

StatementMeta(, 6733142a-fee5-435e-80a4-e80ba8830b9c, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c511cd68-f1c6-4f15-a889-11220a0223e0)

## Timestamp

In [10]:
df_orderitems = df_orderitems.withColumn("shipping_limit_date", col("shipping_limit_date").cast(TimestampType()))\
                                .withColumn("price", col("price").cast(FloatType()))\
                                .withColumn("freight_value", col("freight_value").cast(FloatType()))

StatementMeta(, 6733142a-fee5-435e-80a4-e80ba8830b9c, 12, Finished, Available, Finished, False)

In [11]:
df_orderitems.printSchema()

StatementMeta(, 6733142a-fee5-435e-80a4-e80ba8830b9c, 13, Finished, Available, Finished, False)

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: float (nullable = true)
 |-- freight_value: float (nullable = true)



## Replace values

In [12]:
df_payments = spark.read.format("csv").option("header","true").option("inferSchema",True).load("abfss://SusilNayakWS@onelake.dfs.fabric.microsoft.com/bronze_lh.Lakehouse/Files/RawData/olist_order_payments_dataset.csv")
# df now is a Spark DataFrame containing CSV data from "abfss://SusilNayakWS@onelake.dfs.fabric.microsoft.com/bronze_lh.Lakehouse/Files/RawData/olist_order_payments_dataset.csv".
display(df_payments.limit(5))

StatementMeta(, 6733142a-fee5-435e-80a4-e80ba8830b9c, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5de2512f-4790-45f5-a5f3-8fbfd8520ebb)

In [13]:
df_payments = df_payments.withColumn("payment_type", regexp_replace(col("payment_type"), "_"," "))
display(df_payments.limit(5))

StatementMeta(, 6733142a-fee5-435e-80a4-e80ba8830b9c, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ce5e078a-1b43-499e-838f-9a349fa1a8e0)

# string transformations

In [14]:
df_reviews = spark.read.format("csv").option("header","true").option("inferSchema",True).load("Files/RawData/olist_order_reviews_dataset.csv")
# df now is a Spark DataFrame containing CSV data from "Files/RawData/olist_order_reviews_dataset.csv".
display(df_reviews)

StatementMeta(, 6733142a-fee5-435e-80a4-e80ba8830b9c, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6d35737b-424f-41fe-9468-e87d3c90fe3a)

In [15]:
df_reviews = df_reviews.withColumn("review_comment_title", upper(col("review_comment_title")))\
                        .withColumn("review_comment_message", lower(col("review_comment_message")))

display(df_reviews)

StatementMeta(, 6733142a-fee5-435e-80a4-e80ba8830b9c, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d14351f0-e3f5-41b0-9cfc-4c1a1f88b844)

## handling null values

In [16]:
df_reviews = df_reviews.fillna({"review_score":0, "review_comment_title": "N/A", "review_comment_message": "N/A"})
display(df_reviews.limit(16))

StatementMeta(, 6733142a-fee5-435e-80a4-e80ba8830b9c, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6c9006e2-9387-4aa1-8d32-24878d09f31c)

## Functions

In [17]:
df_products = spark.read.format("csv").option("header","true").load("Files/RawData/olist_products_dataset.csv")
# df now is a Spark DataFrame containing CSV data from "Files/RawData/olist_products_dataset.csv".
display(df_products)

StatementMeta(, 6733142a-fee5-435e-80a4-e80ba8830b9c, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d12ffdc8-21a1-4b0b-871f-a84c314c9de5)

In [18]:
df_products = df_products.withColumns({
    "product_name_lenght": col("product_name_lenght").cast(IntegerType()),
    "product_description_lenght": col("product_description_lenght").cast(IntegerType()),
    "product_photos_qty": col("product_photos_qty").cast(IntegerType()),
    "product_weight_g": col("product_weight_g").cast(IntegerType()),
    "product_height_cm": col("product_height_cm").cast(IntegerType()),
    "product_width_cm": col("product_width_cm").cast(IntegerType()) # Fixed typo here
})

display(df_products.limit(10))

StatementMeta(, 6733142a-fee5-435e-80a4-e80ba8830b9c, 20, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1c58c60d-b628-4b58-aeb6-43c5d96feef3)

## Filtering


In [19]:
# products with weight > 300g
df_products_300g = df_products.filter(col("product_weight_g") > 300)
display(df_products_300g.limit(10))

StatementMeta(, 6733142a-fee5-435e-80a4-e80ba8830b9c, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 269dc917-ce95-48b4-bf8f-6c5385977394)

## Flagging

In [20]:
df_flg_300 = df_products.withColumn("flag300", when(col("product_weight_g") > 300, "Y").otherwise("N"))
display(df_flg_300)

StatementMeta(, 6733142a-fee5-435e-80a4-e80ba8830b9c, 22, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, cc3c8c39-f1a0-49d1-884f-6842cc431103)

## Drop Null columns

In [21]:
df_orders = spark.read.format("csv").option("header","true").load("Files/RawData/olist_orders_dataset.csv")
# df now is a Spark DataFrame containing CSV data from "Files/RawData/olist_orders_dataset.csv".
display(df_orders)

StatementMeta(, 6733142a-fee5-435e-80a4-e80ba8830b9c, 23, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a1bd4354-6948-4581-8c58-20c11871484c)

In [22]:
df_orders = df_orders.dropna(subset=['order_delivered_carrier_date', 'order_delivered_customer_date'])
display(df_orders)

StatementMeta(, 6733142a-fee5-435e-80a4-e80ba8830b9c, 24, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c4dfd701-a743-4feb-8beb-0408c8b4044b)

## Spark sql

##### We cannot directly write sql command on the dataframe itself we need to first convert it into temporary View.

In [23]:
df_orders.createOrReplaceTempView("order_view")

StatementMeta(, 6733142a-fee5-435e-80a4-e80ba8830b9c, 25, Finished, Available, Finished, False)

In [24]:
%%sql
SELECT * FROM order_view

StatementMeta(, 6733142a-fee5-435e-80a4-e80ba8830b9c, 26, Finished, Available, Finished, False)

<Spark SQL result set with 1000 rows and 8 fields>

In [25]:
## or we can also use
display(spark.sql("select count(*) from  order_view"))

StatementMeta(, 6733142a-fee5-435e-80a4-e80ba8830b9c, 27, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 25519546-82f6-4df7-ab01-67f35282b0ca)

In [26]:
%%sql
SELECT *, row_number() OVER(ORDER BY order_id) as row_number FROM order_view

StatementMeta(, 6733142a-fee5-435e-80a4-e80ba8830b9c, 28, Finished, Available, Finished, False)

<Spark SQL result set with 1000 rows and 9 fields>

In [27]:
## covert sql to df

df_sql = spark.sql("""SELECT *, row_number() OVER(ORDER BY order_id) as row_number FROM order_view""")
display(df_sql.limit(10))

StatementMeta(, 6733142a-fee5-435e-80a4-e80ba8830b9c, 29, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 08583eb8-857b-417c-b3da-9304ca93bb00)

### Write into tables

In [32]:
absolute_path = "abfss://SusilNayakWS@onelake.dfs.fabric.microsoft.com/susil_lakehouse_silver.Lakehouse/Files/RawOrder"

df_sql.write.format("delta")\
            .mode("append")\
            .save(absolute_path)

StatementMeta(, 6733142a-fee5-435e-80a4-e80ba8830b9c, 34, Finished, Available, Finished, False)

#### Writing a managed table


In [35]:
df_sql.write.format("delta")\
            .mode("append")\
            .saveAsTable("susil_lakehouse_silver.dbo.raw_table")

StatementMeta(, 6733142a-fee5-435e-80a4-e80ba8830b9c, 37, Finished, Available, Finished, False)